In [1]:
# WorldCup Copilot 2026 — Fine-tuning Llama3 avec LoRA

## Étape 1 — Installer les dépendances
```python
!pip install transformers peft accelerate bitsandbytes datasets trl -q
```

## Étape 2 — Monter Google Drive
```python
from google.colab import drive
drive.mount('/content/drive')
```

## Étape 3 — Charger le dataset
```python
import json

with open('/content/drive/MyDrive/worldcup/dataset_llama3.json', 'r') as f:
    data = json.load(f)

print(f"Dataset chargé : {len(data)} exemples")
```

## Étape 4 — Charger le modèle avec quantization 4-bit
```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Modèle chargé !")
```

## Étape 5 — Configurer LoRA
```python
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
```

## Étape 6 — Préparer le dataset
```python
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size=0.1)
print(dataset)
```

## Étape 7 — Lancer l'entraînement
```python
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="/content/worldcup-llama3-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    save_steps=50,
    logging_steps=10,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=512,
)

trainer.train()
print("Entraînement terminé !")
```

## Étape 8 — Sauvegarder le modèle
```python
model.save_pretrained("/content/worldcup-llama3-lora")
tokenizer.save_pretrained("/content/worldcup-llama3-lora")

# Copier sur Google Drive
import shutil
shutil.copytree(
    "/content/worldcup-llama3-lora",
    "/content/drive/MyDrive/worldcup/worldcup-llama3-lora"
)
print("Modèle sauvegardé sur Google Drive !")
```

SyntaxError: invalid syntax (3249426868.py, line 4)